CSU Channel Island

In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Abbasi, Bahareh' into 'Bahareh Abbasi'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Examples:
    'Associate Professor - Mechatronics Engineering'
    -> 'Mechatronics Engineering'

    'Lecturer AY - Anthropology - 3'
    -> 'Anthropology'
    """
    if " - " in title:
        parts = title.split(" - ")

        # The department is normally the part after the job title.
        return parts[1].strip()

    # Handles titles such as "Professor of Chemistry"
    match = re.search(
        r"(?:Professor|Lecturer|Instructor)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return None


url = "https://ciapps.csuci.edu/directory/Home?id=Faculty&filters=Faculty&optionsVisible=true"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

faculty = []

for row in soup.find_all("tr"):
    name_cell = row.find("th", scope="row")
    cells = row.find_all("td")

    if name_cell is None or len(cells) < 1:
        continue

    name_link = name_cell.find("a")

    if name_link is None:
        continue

    original_name = name_link.get_text(" ", strip=True)
    title = cells[0].get_text(" ", strip=True)

    faculty.append({
        "name": first_last_name(original_name),
        "department": get_department(title),
        "school": "CSU Channel Islands"
    })

df = pd.DataFrame(faculty)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csuci_faculty.parquet", index=False)

                       name                department               school
0            Bahareh Abbasi  Mechatronics Engineering  CSU Channel Islands
1              Reza Abdolee                  Comp Sci  CSU Channel Islands
2        Leslie Marie Abell                      None  CSU Channel Islands
3   Marc Alexander Abramiuk              Anthropology  CSU Channel Islands
4  Christopher James Acosta      Academic Internships  CSU Channel Islands
Faculty found: 100


CSU Bakersfield

In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Acharya, Tathagata' into 'Tathagata Acharya'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Example:
    'Chair and Associate Professor of Chemistry and Biochemistry'
    -> 'Chemistry and Biochemistry'
    """
    match = re.search(
        r"(?:Professor|Lecturer|Instructor)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    # Some records may not clearly state a department
    return None


url = "https://catalog.csub.edu/general-information/csub-information/faculty_list/"

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

faculty = []

for row in soup.find_all("tr"):
    cells = row.find_all("td")

    if len(cells) != 2:
        continue

    original_name = cells[0].get_text(" ", strip=True)
    title = cells[1].get_text(" ", strip=True)

    faculty.append({
        "name": first_last_name(original_name),
        "department": get_department(title),
        "school": "CSU Bakersfield"
    })

df = pd.DataFrame(faculty)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csub_faculty.parquet", index=False)

                       name                        department           school
0         Tathagata Acharya           Physics and Engineering  CSU Bakersfield
1      Moises Acuna-Gurrola                           History  CSU Bakersfield
2  Mian Ahmed Shaheer Afaqi  Philosophy and Religious Studies  CSU Bakersfield
3            Ankita Agarwal          Management and Marketing  CSU Bakersfield
4                Andy Alali                    Communications  CSU Bakersfield
Faculty found: 498


CSU Northridge


In [4]:
!playwright install --with-deps chromium

Installing dependencies...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.2 MB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:8 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [1,858 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,544 kB]
Get:13 http://security.ubuntu.com/ubuntu noble-security/

In [3]:
!pip install playwright pyarrow
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 18.7 MB/s eta 0:00:00
186.8 MiB [] 0% 384.7s186.8 MiB [] 0% 29.9s186.8 MiB [] 0% 19.6s186.8 MiB [] 0% 16.4s186.8 MiB [] 0% 8.1s186.8 MiB [] 1% 4.4s186.8 MiB [] 2% 3.5s186.8 MiB [] 3% 3.2s186.8 MiB [] 4% 2.8s186.8 MiB [] 5% 2.6s186.8 MiB [] 6% 2.4s186.8 MiB [] 6% 2.6s186.8 MiB [] 7% 2.8s186.8 MiB [] 7% 2.9s186.8 MiB [] 7% 3.1s186.8 MiB [] 7% 3.2s186.8 MiB [] 8% 3.0s186.8 MiB [] 9% 2.8s186.8 MiB [] 10% 2.6s186.8 MiB [] 11% 2.5s186.8 MiB [] 12% 2.4s186.8 MiB [] 13% 2.3s186.8 MiB [] 14% 2.3s186.8 MiB [] 15% 2.2s186.8 MiB [] 16% 2.1s186.8 MiB [] 17% 2.0s186.8 MiB [] 18% 2.0s186.8 MiB [] 19% 2.0s186.8 MiB [] 20% 2.0s186.8 MiB [] 20% 2.1s186.8 MiB [] 21% 2.0s186.8 MiB [] 22% 1.9s186.8 MiB [] 24% 1.9s186.8 MiB [] 24% 2.0s186.8 MiB [] 25% 1.9s186.8 MiB [] 26% 2.1s186.8 MiB [] 27% 2.0s186.8 MiB [] 28% 2.0s186.8 MiB [] 29% 1.9s186.8 MiB [] 30% 1.8s186.8 MiB [] 32% 1.8s186.8 MiB [] 33% 1.7s186.8 MiB [] 34% 1.6s186.8 MiB [] 36% 1.6s186.8 MiB 

In [5]:
import re
import string
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


def first_last_name(name):
    """Turns 'Abdelsayed, Michael' into 'Michael Abdelsayed'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Example:
    'Assistant Professor of Biology'
    -> 'Biology'
    """
    match = re.search(
        r"(?:Professor|Lecturer|Instructor)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return None

async def scrape_csun():
    faculty = []
    failed_letters = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for letter in string.ascii_lowercase:
            url = f"https://catalog.csun.edu/faculty/{letter}/"

            try:
                response = await page.goto(
                    url,
                    wait_until="domcontentloaded"
                )

                if response is None or response.status >= 400:
                    status = response.status if response else "no response"
                    print(f"Could not load {letter.upper()}: {status}")

                    failed_letters.append({
                        "letter": letter.upper(),
                        "url": url,
                        "status": status
                    })
                    continue

                soup = BeautifulSoup(
                    await page.content(),
                    "html.parser"
                )

                people_on_page = 0

                for description in soup.find_all(
                    string=re.compile(r"^\s*\(\d{4}\)")
                ):
                    bio = description.get_text(" ", strip=True)
                    name_link = description.find_previous("a")

                    if name_link is None:
                        continue

                    name = name_link.get_text(" ", strip=True)

                    title_match = re.match(
                        r"^\(\d{4}\)\s*(.*?)\.",
                        bio
                    )

                    if title_match and len(name) > 1:
                        title = title_match.group(1)

                        faculty.append({
                            "name": first_last_name(name),
                            "department": get_department(title),
                            "school": "CSU Northridge"
                        })

                        people_on_page += 1

                print(f"{letter.upper()}: {people_on_page} people found")

                # One-second pause between pages
                await page.wait_for_timeout(1000)

            except Exception as error:
                print(f"Error on {letter.upper()}: {error}")

                failed_letters.append({
                    "letter": letter.upper(),
                    "url": url,
                    "status": str(error)
                })

        await browser.close()

    return faculty, failed_letters

In [6]:
faculty, failed_letters = await scrape_csun()

df = (
    pd.DataFrame(faculty)
    .drop_duplicates(subset=["name", "department"])
    .sort_values("name")
)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csun_faculty.parquet", index=False)

A: 46 people found
B: 78 people found
C: 82 people found
D: 45 people found
E: 23 people found
F: 30 people found
G: 72 people found
H: 74 people found
I: 5 people found
J: 35 people found
K: 55 people found
L: 56 people found
M: 100 people found
N: 27 people found
O: 20 people found
P: 42 people found
Q: 6 people found
R: 59 people found
S: 95 people found
T: 39 people found
U: 1 people found
V: 24 people found
W: 47 people found
X: 1 people found
Y: 14 people found
Z: 15 people found
                    name                  department          school
576    Aaron D. Lindberg                        None  CSU Northridge
208           Aaron Daly              Art and Design  CSU Northridge
101  Abdelaziz Boulesbaa  Chemistry and Biochemistry  CSU Northridge
294          Abel Franco                  Philosophy  CSU Northridge
697    Abhijit Mukherjee      Mechanical Engineering  CSU Northridge
Faculty found: 1091


CSU Los Angeles


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

faculty = []

for i in range(1,13):
  url = f"https://www.calstatela.edu/facultydirectory?page={i}"
  response = requests.get(
      url,
      headers={"User-Agent": "Mozilla/5.0"}
  )
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")



  for row in soup.find_all("tr"):
      cells = row.find_all("td")

      # Faculty rows have: name, department, email
      if len(cells) < 2:
          continue

      name = cells[0].get_text(" ", strip=True)
      department = cells[1].get_text(" ", strip=True)

      # Ignore the table header or blank rows
      if not name or name == "Name":
          continue

      faculty.append({
          "name": name,
          "department": department,
          "school": "CSU Los Angeles"
      })

df = pd.DataFrame(faculty).drop_duplicates(subset=["name"])

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csula_faculty.parquet", index=False)

                  name                                    department  \
0       Best, Sherwood  Department of Special Education & Counseling   
1         Beyer, Robbi                     Department of Kinesiology   
2       Bezdecny, Kris           Geography, Geology, and Environment   
3         Black, Sarah                            Extended Education   
4  Blaszczynski, Carol                      Department of Management   

            school  
0  CSU Los Angeles  
1  CSU Los Angeles  
2  CSU Los Angeles  
3  CSU Los Angeles  
4  CSU Los Angeles  
Faculty found: 480


CSU Long Beach


In [8]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Smith, Jane' into 'Jane Smith'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()

urls = [
     "https://sso.csulb.edu",
 "https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory",
                                              "https://www.csulb.edu/school-of-art/faculty-and-staff",
                 "https://www.csulb.edu/dance/faculty-staff-directory",
             "https://web.csulb.edu/colleges/cota/music/faculty-&-staff/",
                                                     "https://www.csulb.edu/theatre-arts/faculty-staff",
          "https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff",
                                  " https://www.csulb.edu/college-of-education/college-of-education-faculty-staff",
"https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff",
                                        "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
                                                    "https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0",
                               "http://web.csulb.edu/colleges/cba/contact/index.php?dept=1",
                                      "https://web.csulb.edu/colleges/cob/contact/index.php?dept=3",
                                        "https://web.csulb.edu/colleges/cob/contact/index.php?dept=4",
                            "http://web.csulb.edu/colleges/cba/contact/index.php?dept=5",
                                   "https://web.csulb.edu/colleges/cob/contact/index.php?dept=6",
                                          "https://www.csulb.edu/hung-family-college-of-engineering/biomedical-engineering/faculty-staff",
                                "https://www.csulb.edu/hung-family-college-of-engineering/chemical-engineering/faculty-staff",
                      "https://www.csulb.edu/college-of-engineering/civil-engineering-construction-engineering-management/faculty-staff",
                         "https://www.csulb.edu/college-of-engineering/computer-engineering-computer-science/faculty-staff",
           "https://www.csulb.edu/hung-family-college-of-engineering/electrical-engineering/faculty-staff",
                             "https://www.csulb.edu/hung-family-college-of-engineering/mechanical-aerospace-engineering/mae-people",
                                            "https://www.csulb.edu/college-of-health-human-services/health-science/faculty-staff",

                                                   "https://www.csulb.edu/college-of-health-human-services/health-care-administration/faculty-staff",
                        "https://www.csulb.edu/college-of-health-human-services/school-of-social-work/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/kinesiology/faculty-staff-0",
                     "https://www.csulb.edu/college-of-health-human-services/school-of-nursing/faculty-and-staff",
                                                    "https://www.csulb.edu/college-of-health-human-services/speech-language-pathology/faculty-staff",
                                       "https://www.csulb.edu/college-of-health-human-services/physical-therapy/faculty-staff",
                                                  "https://www.csulb.edu/college-of-health-human-services/gerontology/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/child-development-family-studies/faculty",
     "https://www.csulb.edu/college-of-health-human-services/recreation-and-leisure-studies/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/family-and-consumer-sciences/faculty-and-staff",
     "https://www.csulb.edu/college-of-health-human-services/public-policy-and-administration/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/hospitality-management/faculty",
     "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/psychology/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/anthropology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/philosophy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/chicano-and-latino-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/english/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/africana-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-indian-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/religious-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/asl-linguistics-deaf-cultures-program/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/global-studies/core-faculty",
     "https://www.csulb.edu/college-of-liberal-arts/comparative-world-literature-and-classics/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/department-of-history/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/economics/department-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/geography/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/human-development/hdev-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/journalism-public-relations/faculty",
     "https://www.csulb.edu/school-of-art/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/asian-and-asian-american-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/communication-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/modern-jewish-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/linguistics-department/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/political-science/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/romance-german-russian-languages-and-literatures/people",
     "https://www.csulb.edu/college-of-liberal-arts/sociology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/womens-gender-sexuality-studies/wgss-faculty-staff",
     "https://www.csulb.edu/biological-sciences/department-directory",
     "https://www.csulb.edu/chemistry-biochemistry/faculty",
     "https://www.csulb.edu/earth-science/department-directory",
     "https://www.csulb.edu/mathematics-statistics/department-directory",
     "https://www.csulb.edu/physics-astronomy/department-directory",
     "https://www.csulb.edu/science-education/department-directory",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty"

]

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

people = []
failed_pages = []
no_directory_table = []


def clean_column_name(column):
    """Makes column names easier to compare."""
    return re.sub(r"[^a-z]", "", str(column).lower())


def find_column(columns, options):
    """
    Finds a DataFrame column whose cleaned name contains
    one of the requested options.
    """
    for column in columns:
        cleaned = clean_column_name(column)

        for option in options:
            if option in cleaned:
                return column

    return None


for raw_url in urls:
    url = raw_url.strip()

    # This is a login page, not a faculty directory
    if "sso.csulb.edu" in url:
        continue

    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()

    except requests.RequestException as error:
        print(f"Could not load: {url}")
        failed_pages.append({
            "source_url": url,
            "error": str(error)
        })
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    page_people_found = 0

    # Look through every table on the current page
    for table in soup.find_all("table"):
        try:
            table_df = pd.read_html(StringIO(str(table)))[0]
        except ValueError:
            continue

        # Turns multi-level column names into regular text
        table_df.columns = [
            " ".join(map(str, col)) if isinstance(col, tuple) else str(col)
            for col in table_df.columns
        ]

        # Find columns even when their order differs from page to page
        first_name_col = find_column(table_df.columns, ["firstname"])
        last_name_col = find_column(table_df.columns, ["lastname"])
        name_col = find_column(
            table_df.columns,
            ["name", "faculty", "person", "employee"]
        )

        title_col = find_column(
            table_df.columns,
            ["title", "position", "rank", "role"]
        )

        department_col = find_column(
            table_df.columns,
            ["department", "area", "unit", "program"]
        )

        email_col = find_column(
            table_df.columns,
            ["email", "contact"]
        )

        # Skip tables that do not look like people directories
        if name_col is None and not (first_name_col and last_name_col):
            continue

        for _, row in table_df.iterrows():

            # Handles separate First Name / Last Name columns
            if first_name_col and last_name_col:
                name = (
                    f"{row[first_name_col]} {row[last_name_col]}"
                ).strip()

            else:
                name = str(row[name_col]).strip()

            # Skip empty rows and repeated table headers
            if not name or name.lower() in ["name", "nan"]:
                continue
            name = first_last_name(name)
            title = (
                str(row[title_col]).strip()
                if title_col and pd.notna(row[title_col])
                else None
            )

            department = (
                str(row[department_col]).strip()
                if department_col and pd.notna(row[department_col])
                else None
            )

            email = (
                str(row[email_col]).strip()
                if email_col and pd.notna(row[email_col])
                else None
            )

            # Removes phone numbers or extra contact text if email is mixed in
            if email:
                email_match = re.search(
                    r"[\w.\-+]+@[\w.\-]+\.\w+",
                    email
                )
                email = email_match.group(0) if email_match else None

            people.append({
                "name": name,
                "department": department,
                "school": "CSU Long Beach",

            })

            page_people_found += 1

    if page_people_found == 0:
        no_directory_table.append({
            "source_url": url
        })

    print(f"{page_people_found} people found: {url}")


# Main faculty/staff output
if people:
    df = pd.DataFrame(people)

    df = (
        df.drop_duplicates(subset=["name", "department"])
          .sort_values(["department", "name"], na_position="last")
    )

    df.to_parquet("csulb_faculty.parquet", index=False)

    print("\nTotal people found:", len(df))
    print(df.head())

else:
    print("No faculty or staff tables were found.")


# Pages that failed to load
pd.DataFrame(failed_pages).to_csv(
    "csulb_failed_pages.csv",
    index=False
)

# Pages that loaded but do not have a recognizable table
pd.DataFrame(no_directory_table).to_csv(
    "csulb_pages_needing_custom_parser.csv",
    index=False
)

46 people found: https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory
75 people found: https://www.csulb.edu/school-of-art/faculty-and-staff
0 people found: https://www.csulb.edu/dance/faculty-staff-directory
0 people found: https://web.csulb.edu/colleges/cota/music/faculty-&-staff/
0 people found: https://www.csulb.edu/theatre-arts/faculty-staff
28 people found: https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff
115 people found: https://www.csulb.edu/college-of-education/college-of-education-faculty-staff
16 people found: https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff
15 people found: https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff
17 people found: https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0
34 people found: http://web.csulb.edu/colleges/cba/contact/index.php?dept=1
34 people found: https://web.csulb.edu/colleges/cob/contact